# D2 — Benchmark dữ liệu nhóm trên Colab CPU

Drive đã có runtime, runtime-hieu, runtime-kien. Người dùng xác nhận runtime của mình còn thiếu. Notebook mặc định là run bộ phận, đánh dấu blocked_incomplete_group_dataset; không dùng làm benchmark chính thức bốn thành viên. Khi đủ dữ liệu, thêm runtime thứ tư và xác nhận GROUP_DATA_COMPLETE/OFFICIAL_RUN.

Thêm shortcut của thư mục Drive nhóm vào MyDrive nếu cần; kiểm tra đúng tên arXiv và đường dẫn thực. Code D trên GitHub phải chứa phiên bản mới trước khi chạy. Chỉ đo sau khi crawler dừng/Drive đồng bộ, xác nhận Colab CPU. Snapshot copy source sang SSD, giữ input gốc; không gồm PDF/references/progress. Timing không gồm copy/checksum/cài dependency; output ghi Drive.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import subprocess
from google.colab import drive

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/arXiv/code')  # repository with src/ and tests/
# One merged runtime is enough; alternatively list all four separate runtimes.
# Confirm that the merged data includes all four members before the official run.
MEMBER_ROOTS = [
    Path('/content/drive/MyDrive/arXiv/runtime'),
    Path('/content/drive/MyDrive/arXiv/runtime-hieu'),
    Path('/content/drive/MyDrive/arXiv/runtime-kien'),
    # Add the user's fourth runtime here after it is uploaded.
]
# Shared folder: https://drive.google.com/drive/folders/1mNYmJH0l19l5bFYddKB-S82CLgtOxe5t
# MyDrive path/capitalization must match the actual shortcut location in your Drive.
OFFICIAL_RUN = False  # True only after all four members are included
GROUP_DATA_COMPLETE = False  # user confirmed their runtime is still missing (2026-11-07)
RESULTS = Path('/content/drive/MyDrive/arXiv/benchmarks')
ALL_CRAWLS_STOPPED = False  # True only after all crawlers stop and Drive syncs
CPU_ONLY_CONFIRMED = False  # Colab hardware accelerator: None

assert ALL_CRAWLS_STOPPED and CPU_ONLY_CONFIRMED, 'Confirm stable inputs and CPU-only runtime'
assert not OFFICIAL_RUN or GROUP_DATA_COMPLETE, 'Official run requires all four members'
assert len(MEMBER_ROOTS) > 0 and len({p.resolve() for p in MEMBER_ROOTS}) == len(MEMBER_ROOTS)
assert (REPO / 'src/parser/benchmark.py').is_file(), 'Check D repository path/version'
assert all((p / 'data').is_dir() for p in MEMBER_ROOTS), 'Check Drive shortcut and all selected runtime paths'
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'src/requirements.txt'], check=True)
sys.path.insert(0, str(REPO / 'src'))
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN = RESULTS / stamp
SNAPSHOT = Path('/content') / ('m2-input-' + stamp)
assert not RUN.exists() and not SNAPSHOT.exists(), 'Choose a fresh run timestamp'
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Snapshot:', SNAPSHOT, '\nOutput:', RUN)


## Kiểm thử cùng commit

Lưu log; pytest exit code khác 0 thì dừng. BC-04 đã có position và acceptance round-trip Parquet, không còn XFAIL dự kiến. Không bỏ qua FAIL để đo.


In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
log_path = RESULTS / (stamp + '-tests.log')
paths = [str(p) for pattern in ('test_a[123]_*.py', 'test_b[123]_*.py', 'test_b_publication.py',
                            'test_parser*.py', 'test_d[12]_*.py', 'test_d_dataset_snapshot.py')
         for p in sorted(Path('tests').glob(pattern))]
with log_path.open('w', encoding='utf8') as log:
    completed = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-rs', *paths],
                               stdout=log, stderr=subprocess.STDOUT, text=True)
print(log_path.read_text(encoding='utf8'))
completed.check_returncode()


## Gom snapshot và đo toàn nhóm

Chờ copy/checksum hoàn tất trước khi đo. Snapshot failed phải điều tra rồi dùng đường dẫn mới. Test dữ liệu tổng hợp không phải kết quả nhóm.


In [ ]:
from parser.dataset_snapshot import prepare_snapshot
from parser.benchmark import benchmark
from parser.benchmark_report import write_report
import json
import shutil

snapshot = prepare_snapshot(MEMBER_ROOTS, SNAPSHOT)
try:
    summary = benchmark(SNAPSHOT, RUN, include_partial=False, group_data_complete=GROUP_DATA_COMPLETE)
finally:
    if RUN.exists():
        shutil.copy2(SNAPSHOT / 'snapshot.json', RUN / 'dataset-snapshot.json')
report = write_report(RUN, RUN / 'm2-results.md')
print(json.dumps(summary, indent=2))
print('Results chapter:', report)


## Đưa kết quả vào book

Book nguồn: https://github.com/NewB-S2deeC0de/arXiv-report. Các chương M2 đã chuẩn bị trong bản checkout book và patch bàn giao report/m2-book.patch của repo code. Giữ summary.json, environment.json, resources.csv, dataset-snapshot.json, log test và parsed/. Sau run chính thức, chạy parser.benchmark_report với --out trỏ tới m2/results.md của repo book, thêm phân tích rồi build lại.

Dữ liệu thiếu thành viên luôn phải ghi rõ phạm vi. completed không có nghĩa acceptance pass; kiểm tra coverage và validation. Không thay bảng trạng thái chưa đo bằng số từ fixture tổng hợp.
